In [4]:
target_word = "зазеркалье"
hint_cost = 5
npcs={"Шляпник":{"letters":["З","Р"],"questions":[{"type":"choice","q":"Что лучше: сказать то, что думаешь, или думать то, что говоришь?","options":["Сказать то, что думаешь","Думать то, что говоришь","Не говорить вообще"],"correct":1,"hint":"Слова без мысли - пустой звук. Важнее осознанность"},{"type":"choice","q":"Время - он или оно?","options":["Оно","Он","Ни то, ни другое"],"correct":1,"hint":"Тот, кто говорит «оно», никогда с Ним чай не пил)"}],"enemy_cost":10},"Кот":{"letters":["Е","К"],"questions":[{"type":"choice","q":"Коты всегда падают на четыре лапы. А я?","options":["Тоже на четыре","На улыбку","Я вообще не падаю"],"correct":2,"hint":"Чтобы упасть, надо быть целиком. А я могу исчезать"},{"type":"text","q":"Что останется, когда меня не станет? (введите слово)","correct_answers":["улыбка","улыбку"],"hint":"То, что появляется при радости и не исчезает"}],"enemy_cost":15},"Королева":{"letters":["А","Ь"],"questions":[{"type":"choice","q":"В моём саду розы должны быть красными. Почему?","options":["Потому что красный - цвет крови","Потому что это мой цвет","Потому что белые скучные"],"correct":1,"hint":"Какой цвет у червовой масти в картах?"},{"type":"text","q":"Дождь смыл краску с роз. Какой цвет остался? (введите слово)","correct_answers":["белый","белые"],"hint":"Цвет снега и чистого листа."}],"enemy_cost":20}}

scenes=[{"id":1,"name":"Чаепитие","npc":"Шляпник","type":"npc"},{"id":2,"name":"Озеро","npc":"Фея","type":"fairy"},{"id":3,"name":"Лабиринт","npc":"Кот","type":"npc"},{"id":4,"name":"Сад","npc":"Королева","type":"npc"},{"id":5,"name":"Финал", "type":"final"}]

alisa={"монеты":25,"жизни":1,"буквы":[],"союзники":[],"нейтралы":[],"враги":[],"история":[],"победы в боях":0}
npc_results = {}

def get_valid_input(min_val, max_val, prompt="Ваш выбор"):
    while True:
        choice = input(f"{prompt} ({min_val}-{max_val}): ")
        if choice.isdigit():
            num = int(choice)
            if min_val <= num <= max_val:
                return num

        print(f"Введите число от {min_val} до {max_val}")

def get_text_input(prompt="Ваш ответ"):
    while True:
        text = input(f"{prompt}: ").strip().lower()
        if text.isalpha():
            return text
        print("Введите текст, используя только буквы!")

def buy_hint(q_data):
    if alisa["монеты"] >= hint_cost:
        print(f"Подсказка доступна! Стоимость: {hint_cost} монет")
        print("1. Купить подсказку")
        print("2. Пропустить")
        choice = get_valid_input(1, 2)
        if choice == 1:
            alisa["монеты"] -= hint_cost
            print(f"Оплачено! Осталось монет: {alisa['монеты']}")
            print(f"Подсказка: {q_data['hint']}")
            alisa["история"].append(f"Куплена подсказка за {hint_cost} монет")
            return True
    else:
        print(f"Недостаточно монет для подсказки")
    return False

def meet_npc(npc_name):
    data = npcs[npc_name]
    print(f"Встреча: {npc_name}")
    correct_count = 0

    i = 1
    for q in data["questions"]:
        print(f"Вопрос {i}: {q['q']}")
        i += 1

        if q["type"] == "choice":
            k = 1
            for option in q["options"]:
                print(f"  {k}. {option}")
                k += 1
            ans = get_valid_input(1, len(q["options"]))
            ok = (ans - 1 == q["correct"])
        else:
            ans = get_text_input()
            ok = ans in q["correct_answers"]

        if ok:
            print("Верно!")
            correct_count += 1
        elif buy_hint(q):
            print("Попробуйте ещё раз:")
            if q["type"] == "choice":
                ans2 = get_valid_input(1, len(q["options"]))
                ok = (ans2 - 1 == q["correct"])
            else:
                ans2 = get_text_input()
                ok = ans2 in q["correct_answers"]
            if ok:
                print("Теперь верно!")
                correct_count += 1
            else:
                print("Снова неверно!")

    # Определяем исход встречи
    if correct_count == 2:
        status = "Союзник"
        target_list = alisa["союзники"]
        earned_letters = data["letters"]
    elif correct_count == 1:
        status = "нейтрал"
        target_list = alisa["нейтралы"]
        earned_letters = [data["letters"][0]]
    else:
        status = "ВРАГ"
        target_list = alisa["враги"]
        earned_letters = []

    target_list.append(npc_name)
    alisa["буквы"].extend(earned_letters)

    msg = f"Буквы: {', '.join(earned_letters)}" if earned_letters else "Придется откупаться"
    print(f"{npc_name} = {status if status != 'ВРАГ' else 'враг'}! {msg}")

    npc_results[npc_name] = {
    "статус": status,
    "правильных": correct_count,
    "буквы": earned_letters
}
    alisa["история"].append(f"Встреча с {npc_name}: {status}")
    return status


def fairy_scene():
    print("Вы встречаете Фею: «Я потеряла ключ. Не поможешь найти?» 1. Помочь 2. Пройти мимо")
    if get_valid_input(1, 2) == 1:
        alisa["союзники"].append("Фея"); alisa["буквы"].append("Л"); alisa["монеты"] += 10
        print("Фея стала союзником! Вы получили букву: Л и 10 монет")
        alisa["история"].append("Фея = Союзник (буква Л, +10 монет)")
    else:
        print("Вы прошли мимо. Фея грустно вздохнула")
        alisa["история"].append("Фея: отказ")

def final_payoff():
    if not alisa["враги"]:
        print("У вас нет врагов! Вы проходите свободно)")
        return True

    print("Финал: Враги преграждают путь! Нужно откупиться")
    total_cost = sum(npcs[enemy]["enemy_cost"] for enemy in alisa["враги"])
    for enemy in alisa["враги"]: print(f"- {enemy}: {npcs[enemy]['enemy_cost']} монет")
    print(f"Итого к оплате: {total_cost} монет\nВаш баланс: {alisa['монеты']} монет")

    if "Королева" in alisa["враги"]:
        if len(alisa["союзники"]) < 1:
            print("Королева не принимает деньги от одиночек!")
            alisa["жизни"] = 0
            return False
        print(f"Союзники ({len(alisa['союзники'])}) поручились за вас")

    if alisa["монеты"] < total_cost:
        print("Недостаточно монет для откупа!")
        alisa["жизни"] = 0
        return False

    print("1. Да")
    print("2. Нет (и автоматически проиграть)")
    if get_valid_input(1, 2) == 1:
        alisa["монеты"] -= total_cost
        alisa["победы в боях"] += len(alisa["враги"])
        print(f"Оплачено! Осталось: {alisa['монеты']} монет")
        alisa["история"].append(f"Откуп: {total_cost} монет")
        return True

    print("Вы отказались платить")
    alisa["жизни"] = 0
    return False

def final_word_puzzle():
    print("Финальная головоломка, ответ - это то, где оказалась Алиса в одном из фильмов (10 букв)")
    attempts = 0
    max_attempts = 3
    while attempts < max_attempts:
        word_input = get_text_input(f"Попытка {attempts + 1}/{max_attempts}")
        if word_input == target_word.lower():
            print(f"Правильно! Слово: {target_word}")
            alisa["история"].append(f"Слово угадано: {target_word}")
            return True
        attempts += 1
        if attempts < max_attempts:
            print(f"Неверно. Осталось попыток: {max_attempts - attempts}")

    print(f"Попытки закончились. Слово: {target_word}")
    alisa["история"].append("Слово НЕ угадано(")
    return False

def get_ending(word_guessed):
    if alisa["жизни"] <= 0:
        return "Потерявшийся"# проиграл в финале
    if not word_guessed:
        return "Заблудившийся"# слово не угадано

    allies = len(alisa["союзники"])
    wins = alisa["победы в боях"]

    if "Королева" in alisa["союзники"] and allies >= 3:
        return "Королева чудес"# подружилась с >=3, обязательно включая Королеву
    if allies >= 2 and wins == 0:
        return "Дипломат"# много союзников, врагов не было вовсе
    if allies >= 1 and wins >= 1:
        return "Искатель"# есть союзники, но пришлось и откупаться
    if allies == 0 and wins >= 1:
        return "Одиночка"# союзников нет, выжила только благодаря монетам
    return "Пробуждение"

def save_log(filename, ending):
    with open(filename, "w", encoding="utf-8") as f:
        print("Отчет по игре", file=f)
        print(f"Концовка: {ending}", file=f)
        print(f"Жизни: {alisa['жизни']}", file=f)
        print(f"Монеты: {alisa['монеты']}", file=f)
        print("Результаты встреч:", file=f)
        for npc, result in npc_results.items():
                print(f"  {npc}: {result['правильных']}/2, буквы: {result['буквы']}", file=f)
        print(f"Союзники: {', '.join(alisa['союзники'])}", file=f)
        print(f"Враги: {', '.join(alisa['враги'])}", file=f)
        print(f"Победы: {alisa['победы в боях']}", file=f)
        print(f"Буквы: {''.join(alisa['буквы'])}", file=f)
        print(file=f)
        print("Дополнительно:", file=f)
        i = 1
        for record in alisa["история"]:
            print(f"{i}. {record}", file=f)
            i += 1
    print(f"Отчёт сохранён: {filename}")

def main_game():
    print("Цели: собрать союзников, избежать врагов, разгадать финальное слово")
    print("По пути вы встретите персонажей из Страны чудес")
    print("Каждый из них задаст вам 2 вопроса")
    print("В зависимости от ответов персонаж может стать союзником/нейтралом/врагом")
    print("За правильные ответы вы получаете буквы для финальной загадки")
    print("Союзники помогут в финале, а враги потребуют откуп за монеты")
    print("Стартовые параметры:")
    print(f"  Жизни: {alisa['жизни']}")
    print(f"  Монеты: {alisa['монеты']}")
    print(f"  Стоимость подсказки: {hint_cost} монет и её можно купить только по одному разу на вопрос, если достаточно монет")
    print("Внимательно отвечайте на вопросы персонажей и собирайте буквы для финальной головоломки!")


    word_guessed = False
    for scene in scenes:
        if alisa["жизни"] <= 0:
            break
        print(f"Сцена {scene['id']}: {scene['name']}")

        if scene["type"] == "npc":
            meet_npc(scene["npc"])
        elif scene["type"] == "fairy":
            fairy_scene()
        elif scene["type"] == "final":
            if not final_payoff():
                break
            if alisa["жизни"] > 0:
                word_guessed = final_word_puzzle()

    ending = get_ending(word_guessed)
    print(f"Концовка: {ending}\nБуквы: {''.join(alisa['буквы'])}\nСоюзники: {', '.join(alisa['союзники']) or 'Нет'}\nМонеты: {alisa['монеты']}")

    save_log("game_log.txt", ending)

main_game()


Цели: собрать союзников, избежать врагов, разгадать финальное слово
По пути вы встретите персонажей из Страны чудес
Каждый из них задаст вам 2 вопроса
В зависимости от ответов персонаж может стать союзником/нейтралом/врагом
За правильные ответы вы получаете буквы для финальной загадки
Союзники помогут в финале, а враги потребуют откуп за монеты
Стартовые параметры:
  Жизни: 1
  Монеты: 25
  Стоимость подсказки: 5 монет и её можно купить только по одному разу на вопрос, если достаточно монет
Внимательно отвечайте на вопросы персонажей и собирайте буквы для финальной головоломки!
Сцена 1: Чаепитие
Встреча: Шляпник
Вопрос 1: Что лучше: сказать то, что думаешь, или думать то, что говоришь?
  1. Сказать то, что думаешь
  2. Думать то, что говоришь
  3. Не говорить вообще
Ваш выбор (1-3): 3
Подсказка доступна! Стоимость: 5 монет
1. Купить подсказку
2. Пропустить
Ваш выбор (1-2): 2
Вопрос 2: Время - он или оно?
  1. Оно
  2. Он
  3. Ни то, ни другое
Ваш выбор (1-3): 2
Верно!
Шляпник = нейтрал